# Ollama Remote Connection And Model Access Test

This notebook only checks whether the Ollama server at `10.0.0.201` is reachable and whether each listed generation and embedding model can be used successfully.


In [ ]:
from __future__ import annotations

import json
import urllib.error
import urllib.request
from pprint import pprint

OLLAMA_ENDPOINT = "http://10.0.0.201:11434"

generation_models = [
    "gemma4",
    "qwen3.5:9b",
    "medgemma1.5",
]

embedding_models = [
    "qwen3-embedding:0.6b",
    "embeddinggemma:latest",
    "all-minilm:latest",
]

print("OLLAMA_ENDPOINT:", OLLAMA_ENDPOINT)
print("generation_models:")
pprint(generation_models)
print("embedding_models:")
pprint(embedding_models)


In [ ]:
def ollama_post(path: str, payload: dict) -> dict:
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        f"{OLLAMA_ENDPOINT}{path}",
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=120) as response:
        return json.loads(response.read().decode("utf-8"))


def ollama_get(path: str) -> dict:
    with urllib.request.urlopen(f"{OLLAMA_ENDPOINT}{path}", timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))


def list_available_models() -> set[str]:
    payload = ollama_get("/api/tags")
    return {item["name"] for item in payload.get("models", [])}


def test_generation_model(model: str) -> dict:
    try:
        payload = ollama_post(
            "/api/generate",
            {
                "model": model,
                "prompt": "Reply with the single word OK.",
                "stream": False,
                "options": {"num_predict": 8},
            },
        )
        return {
            "model": model,
            "kind": "generation",
            "ok": True,
            "detail": (payload.get("response") or "").strip(),
        }
    except Exception as exc:
        return {
            "model": model,
            "kind": "generation",
            "ok": False,
            "detail": f"{type(exc).__name__}: {exc}",
        }


def test_embedding_model(model: str) -> dict:
    try:
        payload = ollama_post(
            "/api/embed",
            {
                "model": model,
                "input": "Embedding connectivity check.",
            },
        )
        embeddings = payload.get("embeddings") or []
        length = len(embeddings[0]) if embeddings and embeddings[0] else 0
        return {
            "model": model,
            "kind": "embedding",
            "ok": length > 0,
            "detail": f"embedding_length={length}",
        }
    except urllib.error.HTTPError as exc:
        if exc.code != 404:
            return {
                "model": model,
                "kind": "embedding",
                "ok": False,
                "detail": f"HTTPError: {exc}",
            }
        try:
            payload = ollama_post(
                "/api/embeddings",
                {
                    "model": model,
                    "prompt": "Embedding connectivity check.",
                },
            )
            embedding = payload.get("embedding") or []
            length = len(embedding)
            return {
                "model": model,
                "kind": "embedding",
                "ok": length > 0,
                "detail": f"embedding_length={length}",
            }
        except Exception as fallback_exc:
            return {
                "model": model,
                "kind": "embedding",
                "ok": False,
                "detail": f"{type(fallback_exc).__name__}: {fallback_exc}",
            }
    except Exception as exc:
        return {
            "model": model,
            "kind": "embedding",
            "ok": False,
            "detail": f"{type(exc).__name__}: {exc}",
        }


In [ ]:
available_models = list_available_models()
print(f"Server reachable. {len(available_models)} model(s) reported by /api/tags.")

missing_generation = [model for model in generation_models if model not in available_models]
missing_embedding = [model for model in embedding_models if model not in available_models]

print("Missing generation models:", missing_generation or "None")
print("Missing embedding models:", missing_embedding or "None")


In [ ]:
results = []

for model in generation_models:
    results.append(test_generation_model(model))

for model in embedding_models:
    results.append(test_embedding_model(model))

results


In [ ]:
try:
    import pandas as pd

    df = pd.DataFrame(results)
    display(df)
    summary = df.groupby(["kind", "ok"]).size().rename("count").reset_index()
    display(summary)
except ModuleNotFoundError:
    pprint(results)
